# Adaptive Learning Platform - Phase 1

This notebook prototypes accessible text transformation and quiz generation before API wiring. Run the cells from top to bottom.

## 1. SETUP

Imports, environment configuration, the shared LLM wrapper, and sample input.

In [5]:
import json
from dotenv import load_dotenv
load_dotenv()
import re
from typing import Any, Dict, List
import os
import pdfplumber
import requests

try:
    from groq import Groq
except ImportError:
    Groq = None

GROQ_API_KEY = os.getenv("GROQ_API_KEY") or os.getenv("GROQ_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL")
GROQ_CLIENT = None

if GROQ_API_KEY and Groq is not None:
    GROQ_CLIENT = Groq(api_key=GROQ_API_KEY)
    if not GROQ_MODEL:
        available_models = {model.id for model in GROQ_CLIENT.models.list().data}
        preferred_models = (
            "llama-3.3-70b-versatile",
            "llama-3.1-8b-instant",
            "openai/gpt-oss-20b",
            "openai/gpt-oss-120b",
        )
        GROQ_MODEL = next(
            (model for model in preferred_models if model in available_models),
            next((model for model in available_models if "llama" in model or "gpt" in model), None),
        )
    if not GROQ_MODEL:
        raise RuntimeError("No text-generation model is available for this Groq project. Set GROQ_MODEL in .env.")


def call_llm(prompt: str) -> str:
    """Generate text with Groq, or use the local fallback when no key is configured."""
    if GROQ_API_KEY:
        if GROQ_CLIENT is None:
            raise ImportError("Install the groq package in this notebook before using GROQ_API_KEY.")
        request_options = {
            "model": GROQ_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.2,
            "max_completion_tokens": 4096,
        }
        if "QUIZ_JSON" in prompt or "CHUNK_JSON" in prompt:
            request_options["response_format"] = {"type": "json_object"}
        if GROQ_MODEL.startswith("openai/"):
            request_options["reasoning_effort"] = "low"
        response = GROQ_CLIENT.chat.completions.create(**request_options)
        return response.choices[0].message.content or ""

    if "QUIZ_JSON" in prompt:
        return json.dumps({
            "question": "What do plants use photosynthesis to produce?",
            "options": ["Glucose", "Sound", "Salt", "Metal"],
            "answer": "Glucose",
            "explanation": "Photosynthesis produces glucose, which stores chemical energy.",
        })
    if "CHUNK_JSON" in prompt:
        sentences = re.split(r"(?<=[.!?])\s+", prompt.split("TEXT:", 1)[-1].strip())
        return json.dumps({"chunks": [" ".join(sentences[index:index + 3]) for index in range(0, len(sentences), 3)]})
    return (
        "Photosynthesis lets plants make food from sunlight. Chlorophyll captures light energy. "
        "Plants use water and carbon dioxide to produce glucose and release oxygen."
    )

print("Groq SDK available:", Groq is not None)
print("Groq API configured:", bool(GROQ_API_KEY))
print("Groq model:", GROQ_MODEL or "local fallback")

# ── VOICE HELP ─────────────────────────────────────────────────────────────────
# Dedicated prompt for the voice/typed Q&A feature.
# Keep this separate from the transformation prompts so it is easy to tune.
VOICE_HELP_PROMPT = """You are an educational assistant inside an adaptive learning platform.
Answer the learner's request using ONLY the supplied lesson content.
Do not invent facts.
Explain clearly and simply.
Keep the response concise (3-5 sentences where possible).
If the answer cannot be determined from the lesson, say:
  "The lesson does not provide enough information to answer that."

LESSON CONTENT:
{lesson}

LEARNER REQUEST:
{request}"""


def voice_ask(user_request: str, lesson_text: str = None) -> str:
    """Send a learner's spoken/typed request to the LLM, grounded in the lesson.

    Falls back gracefully if the LLM is unavailable.
    """
    context = (lesson_text or "").strip()
    if not context:
        # Use the global SOURCE_TEXT if no override is supplied
        try:
            context = SOURCE_TEXT  # noqa: F821  (defined in the next cell)
        except NameError:
            context = "[Lesson text not yet loaded.]"

    prompt = VOICE_HELP_PROMPT.format(lesson=context, request=user_request.strip())
    try:
        return call_llm(prompt)
    except Exception as exc:
        return f"[PRISM error: {exc}]"


Groq SDK available: True
Groq API configured: True
Groq model: openai/gpt-oss-20b


## 2. PDF EXTRACTION

PDF text extraction uses `pdfplumber`, with a page-level fallback for image-only or otherwise empty pages.

In [6]:
def extract_text_and_tables_from_pdf(filepath: str) -> tuple[str, List[List[List[str]]]]:
    """Extract prose text and tables separately so table rows don't bleed into paragraphs.

    Table regions are detected first via find_tables(), then excluded from the
    text extraction pass using page.filter(), so extract_text() only sees prose.
    """
    pages_text: List[str] = []
    pages_tables: List[List[List[List[str]]]] = []

    try:
        with pdfplumber.open(filepath) as pdf:
            for page_number, page in enumerate(pdf.pages, start=1):
                found_tables = page.find_tables()
                table_bboxes = [t.bbox for t in found_tables]  # (x0, top, x1, bottom)

                def is_inside_a_table(obj, boxes=table_bboxes) -> bool:
                    for (tx0, ttop, tx1, tbottom) in boxes:
                        if obj["x0"] >= tx0 and obj["x1"] <= tx1 and obj["top"] >= ttop and obj["bottom"] <= tbottom:
                            return True
                    return False

                prose_only_page = page.filter(lambda obj: not is_inside_a_table(obj))
                page_text = (prose_only_page.extract_text() or "").strip()
                if not page_text:
                    page_text = f"[No extractable prose text found on page {page_number}; OCR may be required.]"
                pages_text.append(page_text)

                pages_tables.append([t.extract() for t in found_tables])
    except FileNotFoundError:
        raise FileNotFoundError(f"PDF file not found: {filepath}")
    except Exception as error:
        raise RuntimeError(f"Could not extract PDF content from {filepath}: {error}") from error

    return "\n\n".join(pages_text), pages_tables


PDF_PATH = "./sample_input.pdf"

# No fallback: if extraction fails, stop here rather than silently using other text.
SOURCE_TEXT, SOURCE_TABLES = extract_text_and_tables_from_pdf(PDF_PATH)

print("--- Extracted prose text ---")
print(SOURCE_TEXT)

print("\n--- Extracted tables ---")
for page_num, page_tables in enumerate(SOURCE_TABLES, start=1):
    for table_num, table in enumerate(page_tables, start=1):
        print(f"\nPage {page_num}, table {table_num}:")
        for row in table:
            print(row)

print("\nSource words:", len(SOURCE_TEXT.split()))
print("Tables found:", sum(len(t) for t in SOURCE_TABLES))

--- Extracted prose text ---
Chapter 7: Photosynthesis and Energy Flow
Grade 9 Biology · Unit 3: Plant Systems
7.1 What is Photosynthesis?
Photosynthesis is the biochemical process by which green plants, algae, and certain bacteria
convert light energy, typically from the sun, into chemical energy stored in glucose molecules.
This process is fundamental to almost all life on Earth, as it forms the base of most food chains
and is responsible for producing the oxygen that most organisms depend on for cellular
respiration. The overall chemical equation for photosynthesis can be summarized as carbon
dioxide plus water, in the presence of light energy, yielding glucose and oxygen.
The process takes place primarily in the chloroplasts of plant cells, specialized organelles that
contain a green pigment called chlorophyll. Chlorophyll is essential because it absorbs light most
efficiently in the blue and red wavelengths of the visible spectrum, while reflecting green light,
which is why most p

## 3. PROFILE-BASED TRANSFORMATION

Prompt constants are deliberately separate from the transformation function so they can be edited without changing control flow.

In [7]:
DYSLEXIA_PROMPT = """Rewrite the text below for a reader with dyslexia. Use short, simple sentences and clear wording. Preserve every fact, relationship, number, and cause-and-effect detail. Do not add facts. Return only the rewritten text.

TEXT:
{text}"""

COGNITIVE_LOAD_PROMPT = """Split the text below into an ordered JSON object with one key, chunks. The chunks value must be an array of digestible chunks. Each chunk must contain 2 to 4 complete sentences. Preserve all original content and facts, do not summarize, and do not add facts. Return only valid JSON. Include the marker CHUNK_JSON nowhere except in this instruction context.

TEXT:
{text}"""

def _parse_json_response(response: str) -> Any:
    """Remove optional Markdown fences and parse a JSON response."""
    cleaned = re.sub(r"^\s*```(?:json)?\s*|\s*```\s*$", "", response.strip(), flags=re.IGNORECASE)
    return json.loads(cleaned)

def transform_text(text: str, profile: str) -> Dict[str, Any]:
    """Transform text according to an accessibility profile."""
    if profile == "low_vision":
        return {"profile": profile, "text": text, "formatting": {"font_size_multiplier": 1.5, "contrast_mode": "high"}}
    if profile == "dyslexia":
        rewritten = call_llm(DYSLEXIA_PROMPT.format(text=text))
        if not rewritten.strip():
            raise ValueError("Groq returned an empty dyslexia transformation.")
        return {"profile": profile, "text": rewritten.strip()}
    if profile == "cognitive_load":
        response = call_llm(COGNITIVE_LOAD_PROMPT.format(text=text))
        parsed = _parse_json_response(response)
        chunks = parsed.get("chunks") if isinstance(parsed, dict) else parsed
        if not isinstance(chunks, list) or not all(isinstance(chunk, str) and chunk.strip() for chunk in chunks):
            raise ValueError("Cognitive-load response must contain a JSON chunks array of strings.")
        return {"profile": profile, "chunks": chunks}
    raise ValueError("profile must be dyslexia, low_vision, or cognitive_load")

for profile in ("dyslexia", "low_vision", "cognitive_load"):
    print(f"\n--- {profile.upper()} ---")
    print(json.dumps(transform_text(SOURCE_TEXT, profile), indent=2, ensure_ascii=False))


--- DYSLEXIA ---
{
  "profile": "dyslexia",
  "text": "Chapter 7: Photosynthesis and Energy Flow  \nGrade 9 Biology · Unit 3: Plant Systems  \n\n7.1 What is Photosynthesis?  \nPhotosynthesis is a chemical process.  \nGreen plants, algae, and some bacteria do it.  \nThey use light from the sun.  \nThey turn that light into chemical energy.  \nThe energy is stored in glucose molecules.  \nThis process is very important.  \nIt is the base of most food chains.  \nIt makes oxygen.  \nOxygen is needed for cellular respiration.  \n\nThe overall equation is:  \ncarbon dioxide + water + light → glucose + oxygen.  \n\nThe process happens in chloroplasts.  \nChloroplasts are parts of plant cells.  \nThey have a green pigment called chlorophyll.  \nChlorophyll absorbs blue and red light.  \nIt reflects green light.  \nThat is why plants look green.  \n\nPhotosynthesis has two stages.  \nEach stage happens in a different part of the chloroplast.  \n\n7.2 The Two Main Stages  \n\nStage 1: Light‑dep

In [8]:
import ipywidgets as widgets
from IPython.display import display, Javascript

PROFILE_LABELS = {
    "I have dyslexia": "dyslexia",
    "I find long or dense text difficult": "cognitive_load",
    "I need larger, high-contrast text": "low_vision",
}

profile_selector = widgets.Dropdown(
    options=list(PROFILE_LABELS),
    value="I have dyslexia",
    description="I am dealing with:",
    layout=widgets.Layout(width="550px"),
)
text_input = widgets.Textarea(
    value=SOURCE_TEXT,
    description="Text:",
    layout=widgets.Layout(width="700px", height="180px"),
)
generate_button = widgets.Button(description="Generate accessible version", button_style="primary")
transformation_output = widgets.Output()

def render_transformation(result: Dict[str, Any]) -> None:
    """Display the generated result in a form suited to the selected profile."""
    with transformation_output:
        transformation_output.clear_output()
        if result["profile"] == "cognitive_load":
            print("Generated digestible chunks:\n")
            for number, chunk in enumerate(result["chunks"], start=1):
                print(f"{number}. {chunk}\n")
        else:
            print(result["text"])
            if result["profile"] == "low_vision":
                print("\nDisplay settings: larger text (1.5x), high contrast")

def generate_selected_transformation(_button: widgets.Button) -> None:
    """Generate output from the user's selected profile and text."""
    text = text_input.value.strip()
    with transformation_output:
        transformation_output.clear_output()
        if not text:
            print("Please enter some text before generating an accessible version.")
            return
        try:
            selected_profile = PROFILE_LABELS[profile_selector.value]
            render_transformation(transform_text(text, selected_profile))
        except (ValueError, json.JSONDecodeError) as error:
            print(f"Could not generate the transformation: {error}")

generate_button.on_click(generate_selected_transformation)

# ── VOICE / Q&A SECTION ────────────────────────────────────────────────────────
_voice_bridge = widgets.Text(
    value="",
    layout=widgets.Layout(display="none"),
)
_voice_bridge.add_class("prism-voice-bridge")

voice_text_input = widgets.Text(
    placeholder="Or type your question here…",
    layout=widgets.Layout(width="600px"),
)
voice_text_input.add_class("prism-voice-input")

mic_button = widgets.Button(
    description="🎙️ Speak",
    button_style="info",
    layout=widgets.Layout(width="130px"),
    tooltip="Click and speak your question (Chrome / Edge only)",
)
mic_button.add_class("prism-mic-btn")

ask_button = widgets.Button(
    description="💬 Ask",
    button_style="warning",
    layout=widgets.Layout(width="100px"),
    tooltip="Send the typed (or transcribed) question to PRISM",
)
ask_button.add_class("prism-ask-btn")

read_aloud_button = widgets.Button(
    description="🔊 Read Aloud",
    button_style="success",
    layout=widgets.Layout(width="140px", display="none"),
    tooltip="Read the PRISM response aloud using browser speech synthesis",
)
read_aloud_button.add_class("prism-read-aloud-btn")

voice_output = widgets.Output()

VOICE_JS_CODE = """
(function () {
  function setVoiceStatus(html) {
    var el = document.getElementById('prism-voice-status');
    if (el) el.innerHTML = html;
  }

  function getBridgeInput() {
    var el = document.querySelector('.prism-voice-bridge input');
    if (el) return el;
    var inputs = document.querySelectorAll('.widget-text input');
    for (var i = 0; i < inputs.length; i++) {
      var wrapper = inputs[i].closest('.widget-text');
      if (wrapper && (wrapper.style.display === 'none' || wrapper.hidden)) return inputs[i];
    }
    return null;
  }

  function getVisibleInput() {
    var el = document.querySelector('.prism-voice-input input');
    if (el) return el;
    var inputs = document.querySelectorAll('.widget-text input');
    for (var i = 0; i < inputs.length; i++) {
      var wrapper = inputs[i].closest('.widget-text');
      if (wrapper && wrapper.style.display !== 'none' && !wrapper.hidden) return inputs[i];
    }
    return null;
  }

  window.startVoiceRecognition = function () {
    var SpeechRecognition = window.SpeechRecognition || window.webkitSpeechRecognition;
    if (!SpeechRecognition) {
      setVoiceStatus('<span style="color:#ef4444;">⚠️ Speech recognition not supported in this browser. Please use Chrome or Edge, or type below.</span>');
      return;
    }

    setVoiceStatus('<span style="color:#10b981; font-weight:bold;">🎙️ Listening… Speak clearly now!</span>');
    var recog = new SpeechRecognition();
    recog.lang = 'en-US';
    recog.interimResults = false;
    recog.maxAlternatives = 1;

    recog.onresult = function (event) {
      var transcript = event.results[0][0].transcript;
      setVoiceStatus('<span style="color:#0284c7; font-weight:bold;">Heard: "' + transcript + '"</span>');

      var visible = getVisibleInput();
      if (visible) {
        var setter = Object.getOwnPropertyDescriptor(window.HTMLInputElement.prototype, 'value').set;
        setter.call(visible, transcript);
        visible.dispatchEvent(new Event('input', { bubbles: true }));
        visible.dispatchEvent(new Event('change', { bubbles: true }));
      }

      var bridge = getBridgeInput();
      if (bridge) {
        var setter = Object.getOwnPropertyDescriptor(window.HTMLInputElement.prototype, 'value').set;
        setter.call(bridge, transcript);
        bridge.dispatchEvent(new Event('input', { bubbles: true }));
        bridge.dispatchEvent(new Event('change', { bubbles: true }));
      } else {
        var askBtn = document.querySelector('.prism-ask-btn');
        if (askBtn) askBtn.click();
      }
    };

    recog.onerror = function (event) {
      if (event.error === 'not-allowed') {
        setVoiceStatus('<span style="color:#ef4444; font-weight:bold;">❌ Microphone permission blocked! Please click the lock or camera/mic icon in your browser URL bar to allow microphone access.</span>');
      } else {
        setVoiceStatus('<span style="color:#f59e0b;">Voice status: ' + event.error + '. Try speaking again or type below.</span>');
      }
    };

    recog.onend = function () {
      var el = document.getElementById('prism-voice-status');
      if (el && el.innerText.includes('Listening')) {
        setVoiceStatus('No speech detected. Click 🎙️ Speak again or type below.');
      }
    };

    try {
      recog.start();
    } catch (err) {
      setVoiceStatus('<span style="color:#ef4444;">Error starting microphone: ' + err.message + '</span>');
    }
  };

  window.prismReadAloud = function (text) {
    if (!window.speechSynthesis) {
      setVoiceStatus('Speech synthesis is not supported in this browser.');
      return;
    }
    window.speechSynthesis.cancel();
    var utter = new SpeechSynthesisUtterance(text);
    utter.lang = 'en-US';
    utter.rate = 0.95;
    window.speechSynthesis.speak(utter);
  };

  function attachDomListeners() {
    var buttons = document.querySelectorAll('button');
    for (var i = 0; i < buttons.length; i++) {
      var b = buttons[i];
      if (b.innerText && b.innerText.includes('Speak')) {
        b.onclick = function (e) {
          e.preventDefault();
          e.stopPropagation();
          window.startVoiceRecognition();
        };
      }
      if (b.innerText && b.innerText.includes('Read Aloud')) {
        b.onclick = function (e) {
          e.preventDefault();
          e.stopPropagation();
          if (window._lastAiSpeech) {
            window.prismReadAloud(window._lastAiSpeech);
          }
        };
      }
    }
  }

  attachDomListeners();
  setTimeout(attachDomListeners, 500);
  setTimeout(attachDomListeners, 1500);
  setTimeout(attachDomListeners, 3000);
})();
"""

_last_ai_response = [""]

def _run_voice_ask(question: str) -> None:
    question = question.strip()
    if not question:
        with voice_output:
            voice_output.clear_output()
            print("Please speak or type a question first.")
        return

    with voice_output:
        voice_output.clear_output()
        print(f'You said: "{question}"\n')
        print("PRISM is thinking…")

    lesson = text_input.value.strip() or SOURCE_TEXT
    response = voice_ask(question, lesson_text=lesson)
    _last_ai_response[0] = response

    safe_resp = response.replace("\\", "\\\\").replace("`", "\\`").replace("$", "\\$")
    with voice_output:
        voice_output.clear_output()
        print(f'You said: "{question}"\n')
        print(f"PRISM:\n{response}")
        display(Javascript(f"window._lastAiSpeech = `{safe_resp}`;"))

    read_aloud_button.layout.display = ""

def on_mic_button_click(_btn):
    with voice_output:
        display(Javascript("window.startVoiceRecognition && window.startVoiceRecognition();"))

def on_ask_button_click(_btn):
    question = _voice_bridge.value.strip() or voice_text_input.value.strip()
    _run_voice_ask(question)

def on_read_aloud_click(_btn):
    text = _last_ai_response[0]
    if not text:
        return
    safe = text.replace("\\", "\\\\").replace("`", "\\`").replace("$", "\\$")
    with voice_output:
        display(Javascript(f"window.prismReadAloud && window.prismReadAloud(`{safe}`);"))

def _on_bridge_change(change):
    new_val = change["new"].strip()
    if new_val:
        voice_text_input.value = new_val
        _run_voice_ask(new_val)

_voice_bridge.observe(_on_bridge_change, names="value")

mic_button.on_click(on_mic_button_click)
ask_button.on_click(on_ask_button_click)
read_aloud_button.on_click(on_read_aloud_click)

voice_divider = widgets.HTML("<hr style='margin:20px 0 12px 0; border:none; border-top:2px solid #ccc;'>")
voice_section_label = widgets.HTML(
    "<b style='font-size:1.05em;'>Need help understanding the lesson?</b>"
)
voice_status = widgets.HTML(
    "<div id='prism-voice-status' style='color:#0284c7; font-weight:500; font-size:0.95em; margin:4px 0 6px 0;'>"
    "Press 🎙️ Speak or type a question below."
    "</div>"
)
voice_buttons_row = widgets.HBox(
    [mic_button, ask_button],
    layout=widgets.Layout(gap="10px", margin="8px 0"),
)
read_aloud_row = widgets.HBox(
    [read_aloud_button],
    layout=widgets.Layout(margin="6px 0"),
)

full_ui = widgets.VBox([
    profile_selector,
    text_input,
    generate_button,
    transformation_output,
    voice_divider,
    voice_section_label,
    voice_status,
    voice_buttons_row,
    voice_text_input,
    _voice_bridge,
    voice_output,
    read_aloud_row,
])

display(full_ui)
display(Javascript(VOICE_JS_CODE))


## 4. QUIZ GENERATION

Each chunk gets one LLM call. The parser accepts plain JSON and JSON wrapped in Markdown code fences.

In [9]:
QUIZ_PROMPT = """Create one practice question based only on the chunk below. Adjust phrasing complexity to the learner profile: {profile}. Return valid JSON only with exactly these keys: question, options, answer, explanation. The options value must be an array of exactly 4 strings. The answer must exactly match one option. Do not use information outside the chunk. Include the marker QUIZ_JSON nowhere except in this instruction context.

CHUNK:
{chunk}"""

def generate_quiz(chunk: str, profile: str) -> Dict[str, Any]:
    """Generate and validate one profile-aware quiz question for a text chunk."""
    response = call_llm(QUIZ_PROMPT.format(chunk=chunk, profile=profile))
    quiz = _parse_json_response(response)
    required_keys = {"question", "options", "answer", "explanation"}
    if set(quiz) != required_keys or not isinstance(quiz["options"], list) or len(quiz["options"]) != 4:
        raise ValueError("Quiz response must contain question, four options, answer, and explanation.")
    if quiz["answer"] not in quiz["options"]:
        raise ValueError("Quiz answer must match one of the options.")
    return quiz

cognitive_chunks = transform_text(SOURCE_TEXT, "cognitive_load")["chunks"]
for index, chunk in enumerate(cognitive_chunks[:3], start=1):
    print(f"\n--- QUIZ FOR CHUNK {index} ---")
    print(json.dumps(generate_quiz(chunk, "cognitive_load"), indent=2))


--- QUIZ FOR CHUNK 1 ---
{
  "question": "Which of the following best describes what photosynthesis does in green plants?",
  "options": [
    "It breaks down glucose to release energy",
    "It converts light energy into chemical energy stored in glucose",
    "It uses chemical energy to produce light",
    "It stores light energy in chlorophyll"
  ],
  "answer": "It converts light energy into chemical energy stored in glucose",
  "explanation": "Photosynthesis is the process by which green plants, algae, and certain bacteria use light energy (usually from the sun) to produce glucose, storing energy chemically."
}

--- QUIZ FOR CHUNK 2 ---
{
  "question": "What is the process described in the chunk that forms the base of most food chains and produces oxygen for cellular respiration?",
  "options": [
    "Photosynthesis",
    "Respiration",
    "Fermentation",
    "Glycolysis"
  ],
  "answer": "Photosynthesis",
  "explanation": "The chunk describes the process that forms the base of m

## 5. END-TO-END TEST

This final cell runs raw text through all three profile paths and generates practice questions for the transformed content.

In [10]:
def chunks_for_quiz(transformed: Dict[str, Any]) -> List[str]:
    """Normalize a transformed profile result into quiz-ready chunks."""
    if transformed["profile"] == "cognitive_load":
        return transformed["chunks"]
    return [transformed["text"]]

# Section 5 end-to-end test
pipeline_summary: Dict[str, Any] = {}
for profile in ("dyslexia", "low_vision", "cognitive_load"):
    transformed = transform_text(SOURCE_TEXT, profile)
    quiz_results = [generate_quiz(chunk, profile) for chunk in chunks_for_quiz(transformed)[:3]]
    pipeline_summary[profile] = {
        "transformed_keys": list(transformed.keys()),
        "quiz_count": len(quiz_results),
        "first_question": quiz_results[0]["question"] if quiz_results else None,
    }

print("\n=== FINAL PIPELINE SUMMARY ===")
print(json.dumps(pipeline_summary, indent=2))


=== FINAL PIPELINE SUMMARY ===
{
  "dyslexia": {
    "transformed_keys": [
      "profile",
      "text"
    ],
    "quiz_count": 1,
    "first_question": "Which part of the chloroplast is where the light\u2011dependent reactions of photosynthesis take place?"
  },
  "low_vision": {
    "transformed_keys": [
      "profile",
      "text",
      "formatting"
    ],
    "quiz_count": 1,
    "first_question": "Which of the following is produced during the light-dependent reactions of photosynthesis?"
  },
  "cognitive_load": {
    "transformed_keys": [
      "profile",
      "chunks"
    ],
    "quiz_count": 3,
    "first_question": "What are the main products of photosynthesis as described in the chunk?"
  }
}
